# Code Ownership, notebook 2: baselines on the development folds

Every baseline is evaluated out of fold: for each of the five development folds it is fitted on the other four (functions never shared). Single-feature baselines are
mapped to a score by a straight line fitted on the training folds; the hand-feature models are tuned with the same nested procedure as the encoder models.

In [1]:
import sys, warnings, json, pickle
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../../helpers").resolve()))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr
import ko_data as K, ko_models as M
from core import metrics, group_bootstrap, paired_diff_bootstrap, qwk, band
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 90)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
d = K.load()
dev = d[d.role == "dev"].reset_index(drop=True)
test = d[d.role == "test"].reset_index(drop=True)
y, groups, folds = dev.score.to_numpy(), dev.snippet_id.to_numpy(), dev.fold.to_numpy()
def ci(p, stat):
    return group_bootstrap(y, p, groups, stat, n=1000)
def sp(a, b):
    return spearmanr(a, b).correlation
def mae(a, b):
    return float(np.mean(np.abs(a - b)))
def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction.text import TfidfVectorizer
H = K.hand_features(dev)
OOF = {}
OOF["training mean"] = np.array([y[folds != f].mean() for f in folds])
def line(x):
    x = np.asarray(x, float).reshape(-1, 1); p = np.zeros(len(y))
    for f in sorted(set(folds)):
        te = folds == f
        p[te] = np.clip(LinearRegression().fit(x[~te], y[~te]).predict(x[te]), 0, 1)
    return p
for col, name in (("n_words", "answer length (words)"), ("identifier_recall", "identifier recall"), ("fact_overlap", "overlap with all facts"), ("focus_overlap", "overlap with the asked-about fact")):
    OOF[name] = line(H[col])
sim = np.zeros(len(y))
for f in sorted(set(folds)):
    te = folds == f
    tf = TfidfVectorizer().fit(pd.concat([dev.answer[~te], dev.reference[~te]]))
    A, R = tf.transform(dev.answer), tf.transform(dev.reference)
    sim[te] = np.asarray(A[te].multiply(R[te]).sum(1)).ravel()
OOF["TF-IDF similarity to the reference"] = line(sim)
for head in ("ridge", "forest", "boosting"):
    OOF[f"hand-built features, {head}"], _ = M.nested(head, H.to_numpy(float), y, groups, folds, workers=10)
pickle.dump(OOF, open(RESULTS / "baselines_oof.pkl", "wb"))
rows = []
for name, p in OOF.items():
    m = metrics(y, p)
    rows.append({"baseline": name, "MAE": m["MAE"], "RMSE": m["RMSE"], "Spearman": m["spearman"], "Pearson": m["pearson"], "QWK (rounded to 1-5)": m["band_qwk"]})
base = pd.DataFrame(rows).set_index("baseline").round(3)
base.to_csv(RESULTS / "baselines.csv")
base

,MAE,RMSE,Spearman,Pearson,QWK (rounded to 1-5)
baseline,,,,,
training mean,0.319,0.367,-0.025,-0.026,0.000
answer length (words),0.309,0.357,0.446,0.240,0.040
identifier recall,0.314,0.353,0.331,0.276,0.078
overlap with all facts,0.323,0.362,0.245,0.181,0.018
overlap with the asked-about fact,0.322,0.362,0.185,0.168,0.000
TF-IDF similarity to the reference,0.317,0.367,0.021,0.042,0.000
"hand-built features, ridge",0.215,0.277,0.637,0.657,0.620
"hand-built features, forest",0.206,0.274,0.649,0.667,0.585
"hand-built features, boosting",0.214,0.281,0.628,0.645,0.562


Intervals (grouped bootstrap over functions, 1,000 resamples) for the two reference points:

In [3]:
for name in ("answer length (words)", "hand-built features, ridge"):
    p = OOF[name]
    print(name, "Spearman", np.round(ci(p, lambda a, b: sp(a, b)), 3), "MAE", np.round(ci(p, mae), 3))

answer length (words) Spearman [0.355 0.573] MAE [0.289 0.326]


hand-built features, ridge Spearman [0.567 0.704] MAE [0.198 0.233]


## Reading the results

**Single features are weak, and several together are not.** Development folds, out of fold: predicting the mean gives MAE 0.319; answer length alone MAE 0.309 (Spearman 0.45); identifier recall 0.314 (0.33); overlap with the asked-about fact 0.322 (0.19); TF-IDF similarity to the reference explanation 0.317 (0.02). The 12 hand-built features together reach MAE 0.206 to 0.215 and Spearman 0.63 to 0.65 (forest, boosting, ridge), QWK 0.56 to 0.62. That is the bar the encoder models have to beat.

TF-IDF similarity to the reference has no signal: the reference explanation is built from the syntax tree in a stiff style that students (and the assistant) do not echo.
